In [ ]:
!pip install tensorflow

In [ ]:
!pip install torch torchvision

In [ ]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.1 MB/s eta 0:00:00


In [ ]:
!pip install tenseal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 60.4 MB/s eta 0:00:00


In [ ]:


import os
import argparse
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from sklearn.preprocessing import MinMaxScaler

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

def load_medmnist(dataset_name: str):

    try:
        import medmnist
        from medmnist import INFO
    except ImportError:
        raise ImportError("Run:  pip install medmnist")

    info      = INFO[dataset_name]
    DataClass = getattr(medmnist, info["python_class"])

    print(f"\nDownloading / loading {dataset_name.upper()} ...")
    train_ds = DataClass(split="train", download=True)
    val_ds   = DataClass(split="val",   download=True)
    test_ds  = DataClass(split="test",  download=True)

    def extract(ds):
        imgs   = ds.imgs.astype("float32") / 255.0
        labels = ds.labels.squeeze().astype("int32")
        if imgs.ndim == 3:                       
            imgs = imgs[..., np.newaxis]
        return imgs, labels

    x_train, y_train = extract(train_ds)
    x_val,   y_val   = extract(val_ds)
    x_test,  y_test  = extract(test_ds)

    print(f"{'='*50}")
    print(f"Dataset : {dataset_name.upper()}")
    print(f"  Train : {x_train.shape}  |  labels {y_train.shape}")
    print(f"  Val   : {x_val.shape}  |  labels {y_val.shape}")
    print(f"  Test  : {x_test.shape}  |  labels {y_test.shape}")
    print(f"  Classes : {info['label']}")
    print(f"{'='*50}\n")

    return x_train, y_train, x_val, y_val, x_test, y_test


def build_autoencoder(input_shape=(28, 28, 1)):


    # Encoder
    inp = layers.Input(shape=input_shape, name="encoder_input")

    x = layers.Conv2D(16, (3,3), activation="relu",
                      padding="valid", name="enc_conv1")(inp)      
    x = layers.MaxPooling2D((2,2), name="enc_pool1")(x)   

    x = layers.Conv2D(8, (3,3), activation="relu",
                      padding="same", name="enc_conv2")(x)         
    x = layers.MaxPooling2D((2,2), padding="same",
                             name="enc_pool2")(x)                

    x = layers.Conv2D(8, (3,3), activation="relu",
                      padding="valid", name="enc_conv3")(x)     
    x = layers.Cropping2D(cropping=((0,1),(0,1)),
                           name="enc_crop")(x)                   
    x = layers.Flatten(name="flatten")(x)                       

    # Decoder
    x = layers.Reshape((4,4,8), name="reshape")(x)       

    x = layers.Conv2D(8, (3,3), activation="relu",
    x = layers.UpSampling2D((2,2), name="dec_up1")(x) 

    x = layers.Conv2D(8, (3,3), activation="relu",
                      padding="same", name="dec_conv2")(x)     
    x = layers.UpSampling2D((2,2), name="dec_up2")(x)   

    x   = layers.Resizing(28, 28, name="dec_resize")(x) 
    out = layers.Conv2D(1, (3,3), activation="sigmoid",
                        padding="same", name="dec_out")(x) 

    autoencoder = Model(inp, out, name="autoencoder")
    return autoencoder


def build_encoder(autoencoder: Model) -> Model:
    """Extract the encoder: input -> 128-D latent vector."""
    return Model(
        inputs  = autoencoder.input,
        outputs = autoencoder.get_layer("flatten").output,
        name    = "encoder"
    )



def train_autoencoder(
    autoencoder : Model,
    x_train     : np.ndarray,
    x_val       : np.ndarray,
    epochs      : int  = 50,
    batch_size  : int  = 32,
    save_path   : str  = "autoencoder_best.keras",
):
    autoencoder.compile(
        optimizer = "adam",
        loss      = "binary_crossentropy",
        metrics   = ["mse"],
    )

    cb_list = [
        callbacks.EarlyStopping(
            monitor              = "val_loss",
            patience             = 10,
            restore_best_weights = True,
            verbose              = 1,
        ),
        callbacks.ModelCheckpoint(
            filepath       = save_path,
            monitor        = "val_loss",
            save_best_only = True,
            verbose        = 1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor  = "val_loss",
            factor   = 0.5,
            patience = 5,
            verbose  = 1,
        ),
    ]

    history = autoencoder.fit(
        x_train, x_train,
        epochs          = epochs,
        batch_size      = batch_size,
        validation_data = (x_val, x_val),
        callbacks       = cb_list,
        verbose         = 1,
    )
    return history


def extract_and_save(
    encoder      : Model,
    splits       : dict,
    dataset_name : str,
    out_dir      : str = ".",
):

    os.makedirs(out_dir, exist_ok=True)

    print("\nExtracting latent features ...")
    raw = {}
    for split_name, (x, _) in splits.items():
        raw[split_name] = encoder.predict(x, verbose=0)
        print(f"  {split_name:5s}: {raw[split_name].shape}")

    scaler = MinMaxScaler()
    raw["train"] = scaler.fit_transform(raw["train"])
    for s in ("val", "test"):
        raw[s] = scaler.transform(raw[s])

    n_feat    = raw["train"].shape[1]        
    feat_cols = [f"feat_{i}" for i in range(n_feat)]

    dfs = {}
    for split_name, (_, y) in splits.items():
        df          = pd.DataFrame(raw[split_name], columns=feat_cols)
        df["label"] = y
        csv_path    = os.path.join(out_dir, f"{dataset_name}_{split_name}_features.csv")
        df.to_csv(csv_path, index=False)
        print(f"  Saved -> {csv_path}  ({df.shape[0]} rows x {df.shape[1]} cols)")
        dfs[split_name] = df

    return dfs



def main(args):


    x_train, y_train, x_val, y_val, x_test, y_test = load_medmnist(args.dataset)

    
    autoencoder = build_autoencoder(input_shape=(28, 28, 1))
    autoencoder.summary()

    encoder = build_encoder(autoencoder)
    encoder.summary()

    weights_path = f"{args.dataset}_autoencoder_best.keras"

    if args.load_weights and os.path.exists(weights_path):
        print(f"\nLoading saved weights from {weights_path} ...")
        autoencoder.load_weights(weights_path)
    else:
        print("\nTraining autoencoder ...")
        train_autoencoder(
            autoencoder,
            x_train, x_val,
            epochs     = args.epochs,
            batch_size = args.batch_size,
            save_path  = weights_path,
        )

    splits = {
        "train": (x_train, y_train),
        "val"  : (x_val,   y_val),
        "test" : (x_test,  y_test),
    }

    dfs = extract_and_save(
        encoder,
        splits,
        dataset_name = args.dataset,
        out_dir      = args.out_dir,
    )

    print("\n-- First 3 rows of train features --")
    print(dfs["train"].head(3).to_string())
    print(f"\nFeature dimensions : {dfs['train'].shape[1] - 1}")
    print(f"Label distribution (train):\n"
          f"{dfs['train']['label'].value_counts().sort_index().to_string()}")
    print("\nAll done!")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Autoencoder feature extraction to CSV "
                    "(BreastMNIST / PneumoniaMNIST)"
    )
    parser.add_argument(
        "--dataset",
        type    = str,
        default = "breastmnist",
        choices = ["breastmnist", "pneumoniamnist"],
        help    = "MedMNIST dataset (default: breastmnist)",
    )
    parser.add_argument(
        "--epochs",
        type    = int,
        default = 50,
        help    = "Max training epochs (default: 50)",
    )
    parser.add_argument(
        "--batch_size",
        type    = int,
        default = 32,
        help    = "Batch size (default: 32)",
    )
    parser.add_argument(
        "--out_dir",
        type    = str,
        default = "outputs",
        help    = "Output directory for CSV files (default: outputs/)",
    )
    parser.add_argument(
        "--load_weights",
        action  = "store_true",
        help    = "Skip training and load saved weights",
    )

    args, _ = parser.parse_known_args()  
    main(args)


Dataset : BREASTMNIST
  Train : (546, 28, 28, 1)  |  labels (546,)
  Val   : (78, 28, 28, 1)  |  labels (78,)
  Test  : (156, 28, 28, 1)  |  labels (156,)
  Classes : {'0': 'malignant', '1': 'normal, benign'}



Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_conv1 (Conv2D)              │ (None, 26, 26, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_pool1 (MaxPooling2D)        │ (None, 13, 13, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_conv2 (Conv2D)              │ (None, 13, 13, 8)      │         1,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_pool2 (MaxPooling2D)        │ (None, 7, 7, 8)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_conv3 (Conv2D)              │ (None, 5, 5, 8)        │           584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_crop (Cropping2D)           │ (None, 4, 4, 8)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 4, 4, 8)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_conv1 (Conv2D)              │ (None, 4, 4, 8)        │           584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_up1 (UpSampling2D)          │ (None, 8, 8, 8)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_conv2 (Conv2D)              │ (None, 8, 8, 8)        │           584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_up2 (UpSampling2D)          │ (None, 16, 16, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_resize (Resizing)           │ (None, 28, 28, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_out (Conv2D)                │ (None, 28, 28, 1)      │            73 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,145 (12.29 KB)

 Trainable params: 3,145 (12.29 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_conv1 (Conv2D)              │ (None, 26, 26, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_pool1 (MaxPooling2D)        │ (None, 13, 13, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_conv2 (Conv2D)              │ (None, 13, 13, 8)      │         1,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_pool2 (MaxPooling2D)        │ (None, 7, 7, 8)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_conv3 (Conv2D)              │ (None, 5, 5, 8)        │           584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_crop (Cropping2D)           │ (None, 4, 4, 8)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 128)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,904 (7.44 KB)

 Trainable params: 1,904 (7.44 KB)

 Non-trainable params: 0 (0.00 B)


Training autoencoder ...
Epoch 1/50
16/18 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.6779 - mse: 0.0626
Epoch 1: val_loss improved from None to 0.65010, saving model to breastmnist_autoencoder_best.keras

Epoch 1: finished saving model to breastmnist_autoencoder_best.keras
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 57ms/step - loss: 0.6636 - mse: 0.0572 - val_loss: 0.6501 - val_mse: 0.0475 - learning_rate: 0.0010
Epoch 2/50
17/18 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.6384 - mse: 0.0448
Epoch 2: val_loss improved from 0.65010 to 0.62108, saving model to breastmnist_autoencoder_best.keras

Epoch 2: finished saving model to breastmnist_autoencoder_best.keras
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - loss: 0.6302 - mse: 0.0413 - val_loss: 0.6211 - val_mse: 0.0352 - learning_rate: 0.0010
Epoch 3/50
17/18 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.6055 - mse: 0.0306
Epoch 3: val_loss improved from 0.62108 to 0.58364, saving model to breastmnist_autoencoder_best.keras

Epoch 3: finished saving mo